# 02 — Scrape CSD MOF Subset Metadata

**Purpose:** Extract crystal structure metadata from the Cambridge Structural Database (CSD) MOF subset, curate synonyms, and deduplicate entries.

**Requires:** CSD Python API licence (v3.6.1+) — run in the CSD Python kernel.

**Inputs:**
- CSD MOF subset database (`MOF_subset.gcd`, bundled with CSD installation)

**Outputs:**
- `data/step-00.csv` — raw CSD MOF entries (identifier, formula, disorder, year, DOI, synonyms)
- `data/step-01.csv` — entries with cleaned synonyms (adsorbate names removed, invalid entries excluded)
- `data/step-02.csv` — final curated list (max 3 identical MOFs per paper to avoid in-situ studies flooding the dataset)

## 1. Extract MOF Metadata from CSD

Iterates over all entries in the CSD MOF subset (~135k structures), extracting the identifier, CCDC number, molecular formula, disorder flag, publication year, DOI, and synonym list. Takes approximately 10 minutes.

In [ ]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import ccdc, ccdc.io, ccdc.search

print("ccdc version:", ccdc.__version__)
print("CSD version:", ccdc.io.csd_version())

subset_path = Path(ccdc.io.csd_directory()) / "subsets" / "CSD_MOF_subsets" / "MOF_subset.gcd"
n_entries = len(ccdc.io.EntryReader(str(subset_path)))
print(f"MOF subset entries: {n_entries}")

rows = [{} for _ in range(n_entries)]
for i, entry in enumerate(tqdm(ccdc.io.EntryReader(str(subset_path)), total=n_entries)):
    for col in ['identifier', 'ccdc_number', 'formula', 'has_disorder']:
        rows[i][col] = getattr(entry, col)
    rows[i]['publication_year'] = entry.publication.year
    doi = entry.publication.doi
    rows[i]['publication_doi'] = doi.lower() if doi else None
    rows[i]['synonyms_orig'] = list(entry.synonyms)

df = pd.DataFrame(rows)
df.to_csv("data/step-00.csv", index=False)
print(f"Saved: data/step-00.csv ({len(df)} entries)")

## 2. Curate Synonyms

Cleans the raw synonym list by removing:
- Misleading prefixes (catena-, Teaching Subset, DrugBank)
- Generic identifiers ("", "MOF-1", "1", "2")
- Adsorbate names appended to MOF names (e.g., "HKUST-1 methane" → "HKUST-1")

In [ ]:
ADSORBATES_LIST = [
    "carbon", "deuteromethane", "methane", "ethane", "ethene",
    "dinitrogen", "acetylene", "acetaldehyde", "ethylamine",
]

df = pd.read_csv("data/step-00.csv")
df['synonyms'] = None

for i, synonyms_str in enumerate(tqdm(df['synonyms_orig'], desc="Curating synonyms")):
    valid = []
    for name in eval(synonyms_str):
        if any([name.startswith('catena'), name.startswith('Teaching Subset'), name.startswith('DrugBank')]):
            continue
        if name in ["", "MOF-1", "MOF-2", 1, 2, "1", "2"]:
            continue
        parts = name.split(" ")
        if len(parts) > 1 and parts[1] in ADSORBATES_LIST:
            name = parts[0]
        valid.append(name)
    df.at[i, 'synonyms'] = valid

df.to_csv("data/step-01.csv", index=False)
print(f"Entries with valid synonyms: {len(df[df['synonyms'] != '[]'])}")

## 3. Deduplicate by Paper

Limits to 3 CSD entries per (DOI, synonym) combination to prevent in-situ studies (e.g., variable-temperature series with hundreds of structures for the same MOF) from dominating the dataset.

In [ ]:
df = pd.read_csv("data/step-01.csv")
df['note'] = "-"
df = df.sort_values(by=['publication_doi', 'synonyms', 'identifier']).reset_index()

max_same = 3
count_same = 0
for i in tqdm(df.index[1:], desc="Deduplicating"):
    if df.at[i, 'synonyms'] != '[]':
        if (df.at[i, 'publication_doi'] == df.at[i-1, 'publication_doi']
                and df.at[i, 'synonyms'] == df.at[i-1, 'synonyms']):
            count_same += 1
        else:
            count_same = 0
        if count_same >= max_same:
            df.at[i, 'note'] = f"Excluding more than {max_same} same MOFs from the same DOI"

df = df.sort_values(by='index').drop(columns='index')
df.to_csv("data/step-02.csv", index=False)

final_count = len(df[(df['synonyms'] != "[]") & (df['note'] == '-')])
print(f"CSD entries with valid synonyms (after dedup): {final_count}")